# aule — The `aule` class (object-oriented API)

Every function in `aule.metrics` and `aule.plots` is also available as a method on the `aule` class, which binds `y_true`/`y_pred` (and optionally `data_format`/`ignore_nan`) once. This is convenient when you're calling many metrics/plots on the same pair of arrays.

Importantly, the class **auto-discovers** every public function in `aule.metrics`/`aule.plots` via introspection — any new metric or plot added to the library automatically becomes available as a method, with no extra code needed.

In [ ]:
!pip install aule

## Standalone functions (functional API)

For reference, here's the same functional-style usage shown in the quickstart.

In [ ]:
import numpy as np
from aule.metrics import rmse, mae, pearson_r, ssim
from aule.plots import plot_field_comparison, plot_scatter

np.random.seed(5)

gt   = np.random.rand(64, 64, 1)
pred = gt + np.random.normal(0, 0.1, gt.shape)

print(rmse(gt, pred))
print(pearson_r(gt, pred))

fig, axes = plot_field_comparison(gt, pred)

# AULE class

Now let's bind `gt`/`pred` once and call everything as methods instead.

In [ ]:
from aule import aule

v = aule(gt, pred)
print(v.rmse())
print(v.pearson_r())
fig, ax = v.plot_scatter(save_path='scatter.png')

In [ ]:
fig, axes = v.plot_field_comparison()

## Every metric works as a method

Including the metrics added after the original release — `mse`, `psnr`, `kge`, etc. — confirming the auto-discovery mechanism picks them up with no extra wiring.

In [ ]:
print('mse: ', v.mse())
print('psnr:', v.psnr())
print('kge: ', v.kge())
print('ssim:', v.ssim())

## `ignore_nan` and `data_format` are bound too

If you construct the `aule` object with `ignore_nan=True` or a specific `data_format`, every method that accepts those parameters will use them automatically — no need to repeat them on every call.

In [ ]:
gt_with_gaps = gt.copy()
gt_with_gaps[:5, :5] = np.nan

v_robust = aule(gt_with_gaps, pred, ignore_nan=True)
print('RMSE (NaNs handled automatically):', v_robust.rmse())

## Methods can still take extra explicit arguments

Functions with additional required parameters (e.g. ensemble-based metrics, which need an extra `y_ensemble` argument) remain fully usable — the bound `y_true` is still injected automatically.

In [ ]:
ensemble = pred[np.newaxis] + np.random.normal(0, 0.05, (10, *pred.shape))

print('CRPS via class:', v.crps(y_ensemble=ensemble))
print('Ensemble spread via class:', v.ensemble_spread(y_ensemble=ensemble))

## Explicit kwargs always override the bound values

If you pass `y_true`/`y_pred` explicitly to a method call, that takes precedence over what was bound at construction time — useful for one-off comparisons against a different array.

In [ ]:
other_pred = gt + np.random.normal(0, 0.3, gt.shape)  # a much noisier alternative prediction

print('Bound prediction RMSE:        ', v.rmse())
print('Overridden prediction RMSE:  ', v.rmse(y_true=gt, y_pred=other_pred))